# CS 195: Natural Language Processing
## PyTorch Embeddings

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmanley/s26-CS195NLP/blob/main/F4_2_PyTorchEmbeddings.ipynb)


## References

[SLP: Embeddings, Chapter 5 of Speech and Language Processing by Daniel Jurafsky & James H. Martin](https://web.stanford.edu/~jurafsky/slp3/5.pdf)

[Word2Vec Tutorial - The Skip-Gram Model by Chris McCormick](http://mccormickml.com/2016/04/19/word2vec-tutorial-the-skip-gram-model/)

[Word2Vec - Negative Sampling made easy by Munesh Lakhey](https://medium.com/@mnshonco/word2vec-negative-sampling-made-easy-9a587cb4695f)

[PyTorch `nn.Embedding` documentation](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html)

[PyTorch `torch.nn.functional.one_hot` documentation](https://pytorch.org/docs/stable/generated/torch.nn.functional.one_hot.html)


In [ ]:
#import sys
#!{sys.executable} -m pip install datasets torch scikit-learn transformers tokenizers

In [5]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
import random 
import torch
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim

def train_tokenizer(sentences, vocabulary_size=200):
    tokenizer = Tokenizer(models.BPE())
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
    trainer = trainers.BpeTrainer(vocab_size=vocabulary_size)

    tokenizer.train_from_iterator(sentences, trainer)
    return tokenizer


def make_skipgrams(sequence, vocabulary_size, window_size=3):
    couples = []
    labels = []

    for i in range(len(sequence)):
        target = sequence[i]
        left = max(0, i - window_size)
        right = min(len(sequence), i + window_size + 1)

        for j in range(left, right):
            if i != j:
                context = sequence[j]

                # positive pair
                couples.append((target, context))
                labels.append(1)

                # generate a random negative pair (in real life, you might want to generate several negative pairs per positive pair)
                negative_context = random.randint(1, vocabulary_size - 1)
                # TODO: we should probably check to make sure this isn't actually a positive pair
                couples.append((target, negative_context))
                labels.append(0)

    return [couples, labels]

def make_skipgrams_batch(tokenized_sentences, vocabulary_size, window_size=3):
    all_couples = []
    all_labels = []
    for tokenized_sentence in tokenized_sentences:
        couples, labels = make_skipgrams(tokenized_sentence, vocabulary_size, window_size)
        all_couples.extend(couples)
        all_labels.extend(labels)

    return [all_couples, all_labels]


def prepare_one_hot_inputs(couples, labels, vocabulary_size):
    inputs = []
    for target_word, context_word in couples:
        target_one_hot = F.one_hot(torch.tensor(target_word), num_classes=vocabulary_size)
        context_one_hot = F.one_hot(torch.tensor(context_word), num_classes=vocabulary_size)
        inputs.append(torch.cat([target_one_hot, context_one_hot]))

    inputs_array = torch.stack(inputs).float()
    labels_array = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

    return inputs_array, labels_array


def train_embedding_model(inputs_array, labels_array, vocabulary_size):

    embedding_model = nn.Sequential(
        nn.Linear(vocabulary_size * 2, 50),
        nn.ReLU(),
        nn.Linear(50, 1)
    )

    loss_fn = nn.BCEWithLogitsLoss() # don't forget - this includes the sigmoid squashing function 
    optimizer = optim.Adam(embedding_model.parameters(), lr=0.0001)

    for epoch in range(20000):
        logits = embedding_model(inputs_array)
        loss = loss_fn(logits, labels_array)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 1000 == 0:
            print(f"epoch {epoch+1}, loss={loss.item():.4f}")

    return embedding_model

def get_embedding(word, tokenizer, embedding_model):
    word_ids = tokenizer.encode(word).ids
    first_layer_weights = embedding_model[0].weight.detach()
    return first_layer_weights[:, word_ids[0]] # use first subword token for this demo



In [ ]:
sentences = [
    "I adopted some dogs from the animal shelter",
    "don't you know that dogs and cats both like scritches",
    "are cats or dogs your favorite animal",
    "I have heard that dogs can be obedient",
    "I have heard that cats can be independent",
    "sharks live in the ocean",
    "many birds fly to get around",
    "dogs and cats are common household pets",
    "cats and dogs both need food and water",
    "my dog and my cat play together",
    "cats and dogs can live in the same home",
    "the puppy and kitten were adopted together",
    "fish swim in the ocean",
    "whales and sharks live in the ocean",
    "boats travel across the ocean",
    "the ocean water is deep and salty",
    "coral reefs are in the ocean"
]

tokenizer = train_tokenizer(sentences, vocabulary_size=200)

tokenized_sentences = []
for sentence in sentences:
    ids = tokenizer.encode(sentence).ids
    tokens = tokenizer.encode(sentence).tokens
    tokenized_sentences.append(ids)

print("Here's an example of some tokens:", tokens)

couples, labels = make_skipgrams_batch(tokenized_sentences, vocabulary_size=tokenizer.get_vocab_size(), window_size=2)

inputs_array, labels_array = prepare_one_hot_inputs(couples, labels, tokenizer.get_vocab_size())

embedding_model = train_embedding_model(inputs_array, labels_array, vocabulary_size=tokenizer.get_vocab_size())





epoch 1000, loss=0.4210
epoch 2000, loss=0.2835
epoch 3000, loss=0.2126
epoch 4000, loss=0.1565
epoch 5000, loss=0.1137
epoch 6000, loss=0.0843
epoch 7000, loss=0.0660
epoch 8000, loss=0.0550
epoch 9000, loss=0.0485
epoch 10000, loss=0.0447
epoch 11000, loss=0.0425
epoch 12000, loss=0.0412
epoch 13000, loss=0.0404
epoch 14000, loss=0.0400
epoch 15000, loss=0.0397
epoch 16000, loss=0.0395
epoch 17000, loss=0.0394
epoch 18000, loss=0.0394
epoch 19000, loss=0.0394
epoch 20000, loss=0.0393


In [8]:

cats_embedding = get_embedding("cats", tokenizer, embedding_model)
dogs_embedding = get_embedding("dogs", tokenizer, embedding_model)
ocean_embedding = get_embedding("ocean", tokenizer, embedding_model)

print(cats_embedding)
print(dogs_embedding)

# distance between dogs and cats should be smaller than distance between dogs and ocean
print(torch.sum((dogs_embedding - cats_embedding) ** 2).item())
print(torch.sum((dogs_embedding - ocean_embedding) ** 2).item())

# let's also look at cosine similarity, which is a common way to measure similarity between vectors that ignores differences in magnitude
# a negative cosine similarity means the vectors are pointing in opposite directions, a positive cosine similarity means they are pointing in the same direction, and a cosine similarity close to 0 means they are orthogonal (i.e. not similar at all)
dogs_cats_similarity = F.cosine_similarity(dogs_embedding, cats_embedding, dim=0).item()
dogs_ocean_similarity = F.cosine_similarity(dogs_embedding, ocean_embedding, dim=0).item()
print("dogs, cats similarity", dogs_cats_similarity)
print("dogs, ocean similarity", dogs_ocean_similarity)


tensor([ 0.8925,  1.2149,  0.6317, -0.7755,  1.1753,  0.8840,  0.7832,  0.9194,
        -0.8472,  0.6898,  1.2281,  0.4809, -0.4792,  0.8787, -1.0552,  0.9823,
         0.9052,  0.8583,  0.6576,  0.7957, -0.5178,  0.9229,  0.7230, -1.0219,
         0.3449,  0.9527,  0.6620, -0.6285,  0.7033, -0.6635,  0.9549,  0.9454,
         0.2980,  0.8383, -1.0567, -0.7198,  1.0091, -0.7191,  1.2245,  0.8676,
        -1.0459,  0.9585,  0.2341,  0.9633,  0.8322, -0.0476,  1.2040,  1.1264,
        -1.0012,  0.9387])
tensor([ 1.0428,  0.9436,  0.1065,  0.3907, -1.1074,  0.8840, -0.8377,  0.4901,
         1.0602, -0.4103,  1.0323, -1.0842,  0.8883,  0.3700,  0.9101,  0.9233,
         0.5463,  0.8620,  0.6262, -0.3732,  0.9156,  0.8524,  0.9121,  1.0005,
        -1.1323,  0.9527, -1.0243,  0.7592, -0.5498, -1.1585,  0.9549,  0.1023,
        -0.9917,  0.9158,  0.0957,  0.9658, -0.7332,  0.6282,  1.0001,  0.8676,
         0.8616,  0.4619,  0.5953, -0.8966,  0.8860,  0.9790,  0.9434,  0.6666,
         0.57

## Dataset for today

AG News dataset
* short news articles
* four classes: World, Sports, Business, Sci/Tech

https://huggingface.co/datasets/fancyzhx/ag_news


In [2]:
from datasets import load_dataset
data = load_dataset("ag_news")

print(data["train"]["text"][0])

# 0 is World
# 1 is Sports
# 2 is Business
# 3 is Sci/Tech
print(data["train"]["label"][0])

Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
2


## Group Exercise

Try creating word embeddings for the AG News dataset. *Note that you will only be able to do a very small subset with this code. Start with 10 examples and work your way up*

* How big of a vocabulary did you need to use to get reasonable tokens for this data?
* Once you use more examples, you'll start to notice that you're going to run out of memory and the kernel will crash. What do you think the issue is? Can you think of a way to do the same idea but keep less data in memory at any given time?



## The PyTorch Embedding Layer
PyTorch provides an `nn.Embedding` layer, which is a linear layer with a lookup table. It allows you to input a token id and look up a row vector (akin to the linear node associated with that id).

It's mathematically equivalent to the one-hot encoding + `nn.Linear` layer we saw before, but it much more memory efficient - we don't have to waste space storing all of those 0s. 

Let's try the same experiment as before but using the `nn.Embedding` layer. 

Note that for this one, we take the dot product 
* (multiply corresponding elements in the vector, add them all up) of the *target* and *context* vectors rather than concatenating their one-hot representations
* product of each dimension contributes towards agreement
* large dot product -> high agreement between the vectors
* `BCEWithLogitsLoss` pushes positive pairs to higher output logits and negative pairs to lower ones
* words appearing with similar contexts get pushed in similar directions


In [ ]:
def train_embedding_model_v2(couples, labels, vocabulary_size):

    embedding_model = nn.Embedding(vocabulary_size, 50)

    loss_fn = nn.BCEWithLogitsLoss() # don't forget - this includes the sigmoid squashing function 
    optimizer = optim.Adam(embedding_model.parameters(), lr=0.001)

    pair_ids = torch.tensor(couples, dtype=torch.long)                # [N, 2]
    labels_tensor = torch.tensor(labels, dtype=torch.float32).unsqueeze(1) # [N, 1]

    for epoch in range(2000):
        target_ids = pair_ids[:, 0]
        context_ids = pair_ids[:, 1]

        target_embeddings = embedding_model(target_ids)
        context_embeddings = embedding_model(context_ids)

        # dot product between target and context embeddings to get a single logit for each pair
        logits = (target_embeddings * context_embeddings).sum(dim=1, keepdim=True)
        loss = loss_fn(logits, labels_tensor )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 100 == 0:
            print(f"epoch {epoch+1}, loss={loss.item():.4f}")

    return embedding_model

def get_embedding_v2(word, tokenizer, embedding_model):
    word_ids = tokenizer.encode(word).ids[0] # use first subword token for this demo
    word_weights = embedding_model.weight.detach()
    return word_weights[word_ids]

sentences = [
    "I adopted some dogs from the animal shelter",
    "don't you know that dogs and cats both like scritches",
    "are cats or dogs your favorite animal",
    "I have heard that dogs can be obedient",
    "I have heard that cats can be independent",
    "sharks live in the ocean",
    "many birds fly to get around",
    "dogs and cats are common household pets",
    "cats and dogs both need food and water",
    "my dog and my cat play together",
    "cats and dogs can live in the same home",
    "the puppy and kitten were adopted together",
    "fish swim in the ocean",
    "whales and sharks live in the ocean",
    "boats travel across the ocean",
    "the ocean water is deep and salty",
    "coral reefs are in the ocean"
]

tokenizer = train_tokenizer(sentences, vocabulary_size=200)

tokenized_sentences = []
for sentence in sentences:
    ids = tokenizer.encode(sentence).ids
    tokens = tokenizer.encode(sentence).tokens
    tokenized_sentences.append(ids)

print(tokens)

couples, labels = make_skipgrams_batch(tokenized_sentences, vocabulary_size=tokenizer.get_vocab_size(), window_size=2)
embedding_model_v2 = train_embedding_model_v2(couples, labels, vocabulary_size=tokenizer.get_vocab_size())




['coral', 'reefs', 'are', 'in', 'the', 'ocean']
epoch 100, loss=1.6345
epoch 200, loss=0.7745
epoch 300, loss=0.4268
epoch 400, loss=0.2964
epoch 500, loss=0.2388
epoch 600, loss=0.2098
epoch 700, loss=0.1885
epoch 800, loss=0.1711
epoch 900, loss=0.1565
epoch 1000, loss=0.1439
epoch 1100, loss=0.1330
epoch 1200, loss=0.1235
epoch 1300, loss=0.1152
epoch 1400, loss=0.1080
epoch 1500, loss=0.1018
epoch 1600, loss=0.0964
epoch 1700, loss=0.0917
epoch 1800, loss=0.0876
epoch 1900, loss=0.0842
epoch 2000, loss=0.0812


In [20]:

cats_embedding = get_embedding_v2("cats", tokenizer, embedding_model_v2)
dogs_embedding = get_embedding_v2("dogs", tokenizer, embedding_model_v2)
ocean_embedding = get_embedding_v2("ocean", tokenizer, embedding_model_v2)

print(cats_embedding)
print(dogs_embedding)

# distance between dogs and cats should be smaller than distance between dogs and ocean
print(torch.sum((dogs_embedding - cats_embedding) ** 2).item())
print(torch.sum((dogs_embedding - ocean_embedding) ** 2).item())

# let's also look at cosine similarity, which is a common way to measure similarity between vectors that ignores differences in magnitude
# a negative cosine similarity means the vectors are pointing in opposite directions, a positive cosine similarity means they are pointing in the same direction, and a cosine similarity close to 0 means they are orthogonal (i.e. not similar at all)
dogs_cats_similarity = F.cosine_similarity(dogs_embedding, cats_embedding, dim=0).item()
dogs_ocean_similarity = F.cosine_similarity(dogs_embedding, ocean_embedding, dim=0).item()
print("dogs, cats similarity", dogs_cats_similarity)
print("dogs, ocean similarity", dogs_ocean_similarity)

tensor([-0.0877, -0.3518, -0.3600,  0.9118, -0.0314,  0.3788, -0.2943, -0.1570,
        -0.0397,  0.0406, -0.1221, -0.3082, -0.7472, -0.2803, -0.0538,  0.1821,
         0.2967, -0.2064, -0.2425,  0.3428,  0.3597,  0.4040,  0.0532, -0.2162,
         0.3885,  0.5960,  0.2533,  0.4583,  0.2333, -0.4344,  0.3727, -0.7224,
        -0.2756,  0.1612,  0.1429, -0.4788, -0.7982,  0.5549,  0.3319, -0.2805,
         0.0786, -0.1142,  0.8001,  0.4621, -0.2386, -0.5024,  0.3182, -0.6769,
        -0.5640, -0.4438])
tensor([ 0.1393,  0.0635, -0.0204,  0.0029, -0.1696, -0.5133, -0.4018, -0.2243,
         0.1644,  0.6108,  0.2779, -0.3973, -0.0749, -0.6295, -0.3006,  0.1332,
        -0.0105,  0.3627,  0.0221, -0.1748,  0.2732,  0.0767, -0.6238, -0.0515,
         0.4343,  0.0975,  0.4711,  0.0826,  0.2348,  0.1738,  0.4205, -0.3538,
        -0.3330,  0.1320, -0.7208, -0.4733, -0.0486,  0.3468,  0.0138, -0.4849,
         0.4425,  0.1617,  0.5394,  0.3862, -0.4646, -0.3649,  0.3832,  0.0577,
         0.06

## Applied Exploration

Create word embeddings for a larger portion of the AG News dataset, say 5000 texts

Show some example word embeddings for some words that appear in the dataset (*cats* and *dogs* may not be good examples for this one)

Describe your results and reflect on them
* How big of a vocabulary did you need to use?
* What learning rate and number of training epochs do you think are appropriate? Why?
* How could you go about figuring out if these embeddings are useful?

## Adding an Embedding layer to your model for other learning tasks

First, let's prepare the data
* We could use the same kind of tokenizer, or just use an existing one - let's go back to doing it with a pretrained Hugging Face tokenizer


In [20]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

data = load_dataset("ag_news")

train_texts = data["train"]["text"]
test_texts  = data["test"]["text"]
train_labels = data["train"]["label"]
test_labels = data["test"]["label"]

hf_tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
hf_tokenizer.add_special_tokens({'pad_token': '[PAD]'}) # for some reason this tokenizer doesn't have a pad token by default, but we need one to be able to batch our inputs together, so we'll just add a new special token for padding

tokenized_train_texts = hf_tokenizer(list(train_texts), truncation=True, padding=True, max_length=128, return_tensors="pt")
tokenized_test_texts = hf_tokenizer(list(test_texts), truncation=True, padding=True, max_length=128, return_tensors="pt")

X_train = tokenized_train_texts["input_ids"]
X_test = tokenized_test_texts["input_ids"]

y_train = torch.tensor(np.array(train_labels), dtype=torch.long)
y_test = torch.tensor(np.array(test_labels), dtype=torch.long)


## Model with an Embedding layer

We'll set up the embedding layer and sequential model as before

The optimizer needs to have a concatenation of the parameters for both parts

In [21]:
embedding = nn.Embedding(len(hf_tokenizer), 50) # len(hf_tokenizer) is the vocab size including the new padding token
classifier = nn.Sequential(
    nn.Linear(50, 100),
    nn.ReLU(),
    nn.Linear(100, 4)
)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(list(embedding.parameters()) + list(classifier.parameters()), lr=0.01)

## Training Loop with Embedding layer

The input goes into the embedding layer
* returns an embedding for each words, so we aggregate them with the *mean* to get an embedding for the whole training example

In [22]:


for epoch in range(100):
    optimizer.zero_grad()

    emb = embedding(X_train) # [batch_size, seq_len, embedding_dim]
    pooled_emb = emb.mean(dim=1) # simple way to get a single vector [batch_size, embedding_dim] for the whole sequence - just average the token embeddings
    logits = classifier(pooled_emb)
    loss = loss_fn(logits, y_train)

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch+1}, loss = {loss.item():.4f}")



Epoch 1, loss = 1.3904
Epoch 11, loss = 1.3370
Epoch 21, loss = 1.0798
Epoch 31, loss = 0.5989
Epoch 41, loss = 0.3742
Epoch 51, loss = 0.2744
Epoch 61, loss = 0.2105
Epoch 71, loss = 0.1677
Epoch 81, loss = 0.1361
Epoch 91, loss = 0.1113


## Evaluating works similarly

In [23]:
with torch.no_grad():
    emb = embedding(X_test)
    pooled_emb = emb.mean(dim=1)
    logits = classifier(pooled_emb)
    predicted_labels = logits.argmax(dim=1)
    accuracy = (predicted_labels == y_test).float().mean()

print("Accuracy:", accuracy.item())

Accuracy: 0.9048684239387512


## Applied Exploration

Perform a text classification experiment with another classification data set
* Try different embedding vector lengths (other than just 50 as we did here)
* Experiment with different neural network structures, learning rates, and number of epochs

Report your results and reflect on them
* Describe your dataset
* Describe what you did
* Report the results you observed
* Discuss any interesting insights